# KAMP PassOrFail Prediction — Concise Time-Aware ML Pipeline

This notebook keeps the machine-learning stage intentionally short because it is the first step before later XAI and causality analysis.

Core choices:

1. Use a **chronological train/validation/test split** because the records are time-indexed manufacturing observations.
2. Exclude leakage/identifier columns such as `Reason`, `_id`, raw dates, and serial identifiers.
3. Compare only three strong state-of-the-art tabular models: **CatBoost**, **LightGBM**, and **XGBoost**.
4. Report only three main metrics suitable for the rare `Fail` class.


## Main evaluation metrics

| Metric | Role in this experiment | Why it is used |
|---|---|---|
| **PR-AUC / Average Precision** | Primary ranking metric | Best summary metric for rare failure detection because it focuses on precision-recall behavior rather than majority-class accuracy. |
| **Fail Recall** | Failure-detection sensitivity | Measures how many true failures are detected. Important because missed defective products are usually more costly than extra inspections. |
| **Fail F1-score** | Precision-recall balance | Provides a compact single-number balance between catching failures and avoiding too many false alarms. |

Accuracy is deliberately not used as a main metric because the dataset is highly imbalanced and a model can look accurate while missing most failures.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
from dotenv import load_dotenv
from openai import OpenAI


from pyvis.network import Network
from IPython.display import IFrame, display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, recall_score, f1_score, precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Top three state-of-the-art tabular ML models.
# If these imports fail, run: %pip install xgboost lightgbm catboost
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from lingam import VARLiNGAM, DirectLiNGAM


RANDOM_STATE = 42
TARGET = "PassOrFail"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")


## 1. Load the preprocessed dataset

In [ ]:
DATA_PATH_CANDIDATES = [
    Path("labeled_data_preprocessed.csv"),
    Path("labeled_data_preprocessed(1).csv"),
    Path("../data/interim/labeled_data_preprocessed.csv"),
    Path("../data/processed/labeled_data_preprocessed.csv"),
    Path("/mnt/data/labeled_data_preprocessed.csv"),
    Path("/mnt/data/labeled_data_preprocessed(1).csv"),
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find labeled_data_preprocessed.csv. Update DATA_PATH_CANDIDATES.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print("Target distribution:")
print(df[TARGET].value_counts().rename(index={0: "Pass", 1: "Fail"}))
print("\nReason distribution among failures:")
print(df.loc[df[TARGET] == 1, "Reason"].value_counts(dropna=False))
df.head()


In [ ]:
df.shape

In [ ]:
df["Reason"].value_counts()

In [ ]:
import pandas as pd

# 1. Ensure the column is in datetime format
df["TimeStamp"] = pd.to_datetime(df["TimeStamp"])

# 2. Filter rows where the date matches "2020-10-27"
df[df["TimeStamp"].dt.date == pd.to_datetime("2020-10-27").date()]["Reason"]

In [ ]:
# df["Reason"].value_counts()

df[df["Reason"]=="초기허용불량"]



# 2020-10-27 00:56:21

## 2. Reason-diverse 80/20 train/test split

For this explanation-evaluation study we instead use a roughly **80% train / 20% test** split (test is smaller than train), while explicitly guaranteeing the test set contains the failure reasons of interest:

1. We **reserve the most recent `N_PER_REASON` (= 5) failures** of each required reason (`가스`, `미성형`, `초기허용불량`) for the test set.
2. The remaining records are split chronologically: the earliest ~80% become **train**, the latest ~20% become the **test tail**.
3. The reserved reason rows are merged into the test set, so `X_test` is guaranteed to contain at least 5 `가스`, 5 `미성형`, and 5 `초기허용불량` failures.

The split stays mostly time-aware (the model is trained on the earliest production records and the bulk of the test set is later data), with the reserved reason rows added so every failure reason of interest is represented in `X_test`.


In [ ]:
if "TimeStamp" not in df.columns:
    raise ValueError("The dataset must contain TimeStamp for a time-aware experiment.")

if "Reason" not in df.columns:
    raise ValueError("The dataset must contain Reason for the explanation-evaluation study.")

time_df = df.copy()
time_df["TimeStamp_dt"] = pd.to_datetime(time_df["TimeStamp"], errors="coerce")
if time_df["TimeStamp_dt"].isna().any():
    raise ValueError("Some TimeStamp values could not be parsed.")

time_df = time_df.sort_values("TimeStamp_dt").reset_index(drop=True)

# ------------------------------------------------------------------
# Reason-diverse 80/20 split
# ------------------------------------------------------------------
# We want roughly an 80%/20% train/test split (test smaller than train),
# while guaranteeing the test set contains at least N_PER_REASON failures
# for each required failure reason for the explanation-evaluation study.
REQUIRED_TEST_REASONS = ["가스", "미성형", "초기허용불량"]
N_PER_REASON = 5
TEST_FRACTION = 0.20

# Reserve the most recent N_PER_REASON failures of each required reason for the test set.
reserved_index = []
for reason in REQUIRED_TEST_REASONS:
    reason_idx = time_df.index[(time_df[TARGET] == 1) & (time_df["Reason"] == reason)]
    if len(reason_idx) < N_PER_REASON:
        raise ValueError(
            f"Need at least {N_PER_REASON} failures with Reason == '{reason}', found {len(reason_idx)}."
        )
    reserved_index.extend(reason_idx[-N_PER_REASON:].tolist())

reserved_index = pd.Index(reserved_index)

# Split the remaining rows chronologically: earliest ~80% -> train, latest ~20% -> test tail.
remaining = time_df.drop(index=reserved_index)
n_test_tail = int(round(len(remaining) * TEST_FRACTION))
n_train = len(remaining) - n_test_tail

train_df = remaining.iloc[:n_train].drop(columns=["TimeStamp_dt"])

# Keep val_df/y_val as empty objects only for compatibility with later cells.
# The experiment now uses only train/test.
val_df = time_df.iloc[0:0].drop(columns=["TimeStamp_dt"])

# Test = chronological tail + reserved required-reason rows, sorted back into time order.
test_df = pd.concat([remaining.iloc[n_train:], time_df.loc[reserved_index]])
test_df = test_df.sort_values("TimeStamp_dt").drop(columns=["TimeStamp_dt"])

split_summary = pd.DataFrame({
    "split": ["train", "validation_removed", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "failures": [int(train_df[TARGET].sum()), int(val_df[TARGET].sum()), int(test_df[TARGET].sum())],
    "start_time": [train_df["TimeStamp"].min(), None, test_df["TimeStamp"].min()],
    "end_time": [train_df["TimeStamp"].max(), None, test_df["TimeStamp"].max()],
})

print("Split summary:")
display(split_summary)

print(f"\nTrain rows: {len(train_df)} ({len(train_df) / len(time_df):.1%}), "
      f"Test rows: {len(test_df)} ({len(test_df) / len(time_df):.1%})")

print("\nFailure reasons in train:")
display(train_df.loc[train_df[TARGET] == 1, "Reason"].value_counts(dropna=False).to_frame("count"))

print("\nFailure reasons in test:")
test_reason_counts = test_df.loc[test_df[TARGET] == 1, "Reason"].value_counts(dropna=False).to_frame("count")
display(test_reason_counts)

# Confirm every required reason is present in the test set with at least N_PER_REASON failures.
for reason in REQUIRED_TEST_REASONS:
    reason_count = int(((test_df[TARGET] == 1) & (test_df["Reason"] == reason)).sum())
    if reason_count < N_PER_REASON:
        raise ValueError(
            f"Expected at least {N_PER_REASON} test failures with Reason == '{reason}', found {reason_count}."
        )
    print(f"Confirmed: X_test/test_df contains {reason_count} rows with Reason == '{reason}'.")


In [ ]:
4174+1058

In [ ]:
df["PassOrFail"].value_counts()

## 3. Feature preparation

`Reason` is removed because it directly reveals why a product failed and is missing for all passed products. Raw timestamps are used for splitting only, not as predictors. Serial/id-like columns are removed to reduce leakage and overfitting risk.


In [ ]:
LEAKAGE_OR_ID_COLUMNS = [
    TARGET,
    "Reason",                 # direct label leakage; kept in test_df only for explanation evaluation metadata
    "_id",                    # row identifier
    "TimeStamp",              # used only for chronological split
    "PART_FACT_PLAN_DATE",    # raw date; avoid date memorization
    "PART_FACT_SERIAL",       # serial/order identifier; avoid memorization
]

def make_xy(data: pd.DataFrame):
    y = data[TARGET].astype(int)
    drop_cols = [c for c in LEAKAGE_OR_ID_COLUMNS if c in data.columns]
    X = data.drop(columns=drop_cols)
    return X, y

X_train, y_train = make_xy(train_df)
X_val, y_val = make_xy(val_df)   # empty; kept only for compatibility with later cells
X_test, y_test = make_xy(test_df)

print("X_train:", X_train.shape, "failures:", int(y_train.sum()))
print("X_val:  ", X_val.shape, "failures:", int(y_val.sum()), "[validation removed]")
print("X_test: ", X_test.shape, "failures:", int(y_test.sum()))

print("\nX_test failure reason distribution:")
display(test_df.loc[test_df[TARGET] == 1, "Reason"].value_counts(dropna=False).to_frame("count"))

X_train.head()


In [ ]:
X_test

In [ ]:
# Candidate failure rows for explanation evaluation.
# These are the rows available in X_test/y_test, with Reason kept in test_df metadata.

test_failure_candidates = (
    test_df.loc[test_df[TARGET] == 1, ["TimeStamp", TARGET, "Reason"]]
    .copy()
)

display(test_failure_candidates)

print("Rows by Reason in test failures:")
display(test_failure_candidates.groupby("Reason").size().to_frame("count"))

# 가스 => Gas
# 미성형 => Short shot / incomplete molding
# 초기허용불량 => Initial allowable / startup defect


## 4. Preprocessing pipeline

The tree-based models do not require scaling. Numeric columns are median-imputed; categorical columns are imputed and one-hot encoded.


In [ ]:
def make_one_hot_encoder():
    # Compatibility with older and newer scikit-learn versions.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(X_frame: pd.DataFrame) -> ColumnTransformer:
    numeric_features = X_frame.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_frame.select_dtypes(exclude=np.number).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


## 5. Three selected models

The selected models are strong, widely used state-of-the-art algorithms for structured/tabular data. Each model is configured for class imbalance.


In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)
print("scale_pos_weight:", round(scale_pos_weight, 2))

model_specs = {
    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        verbose=False,
        random_seed=RANDOM_STATE,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

list(model_specs.keys())


## 6. Evaluation helpers

Because the validation block was removed to make the chronological test set more diverse in `Reason`, the classification threshold below is selected from the training block only. This threshold is used only for diagnostic prediction tables.

For this mini study, the central object is **not** threshold-optimized predictive performance; it is the evaluation of LLM-generated explanations for diverse failure reasons in the chronological test block.


In [ ]:
def get_failure_scores(pipeline: Pipeline, X: pd.DataFrame) -> np.ndarray:
    """Return predicted probability/score for the Fail class.

    We call the final model directly after preprocessing instead of pipeline.predict_proba.
    This avoids occasional scikit-learn compatibility issues with external estimators such as CatBoost.
    """
    Xt = pipeline.named_steps["preprocess"].transform(X)
    model = pipeline.named_steps["model"]
    if hasattr(model, "predict_proba"):
        return model.predict_proba(Xt)[:, 1]
    return model.decision_function(Xt)


def choose_threshold_by_f1(y_true: pd.Series, scores: np.ndarray) -> float:
    """Choose a diagnostic decision threshold.

    In this notebook version, the validation block is removed to obtain a reason-diverse
    chronological test set. Therefore, the threshold is selected from the training block only.
    Do not present this as an independently tuned threshold.
    """
    thresholds = np.linspace(0.01, 0.99, 99)
    rows = []
    for threshold in thresholds:
        pred = (scores >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "fail_f1": f1_score(y_true, pred, zero_division=0),
            "fail_recall": recall_score(y_true, pred, zero_division=0),
        })
    table = pd.DataFrame(rows)
    best = table.sort_values(["fail_f1", "fail_recall"], ascending=False).iloc[0]
    return float(best["threshold"])


def evaluate_three_metrics(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    """Evaluate compact metrics plus confusion counts for interpretation."""
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "PR_AUC": average_precision_score(y_true, scores),
        "Fail_Recall": recall_score(y_true, pred, zero_division=0),
        "Fail_F1": f1_score(y_true, pred, zero_division=0),
        "Fail_Precision_aux": precision_score(y_true, pred, zero_division=0),
        "Threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


## 7. Train on earlier records and evaluate on the reason-diverse chronological test block


In [ ]:
results = []
fitted_models = {}

for model_name, model in model_specs.items():
    print(f"Running {model_name}...")

    pipeline = Pipeline([
        ("preprocess", make_preprocessor(X_train)),
        ("model", clone(model)),
    ])

    # Fit only on the earliest production block.
    pipeline.fit(X_train, y_train)

    # No validation block is used in this version.
    # Threshold is selected on train only and used for diagnostic tables.
    train_scores = get_failure_scores(pipeline, X_train)
    threshold = choose_threshold_by_f1(y_train, train_scores)

    # Evaluate on the later chronological test block containing diverse Reasons.
    test_scores = get_failure_scores(pipeline, X_test)
    metrics = evaluate_three_metrics(y_test, test_scores, threshold)
    metrics["Model"] = model_name
    metrics["Threshold_Source"] = "train_only_diagnostic"

    results.append(metrics)
    fitted_models[model_name] = pipeline

results_df = (
    pd.DataFrame(results)
    .set_index("Model")
    .sort_values(["PR_AUC", "Fail_F1", "Fail_Recall"], ascending=False)
)

results_df.round(4)


## 8. Select model for XAI and explanation-evaluation pipeline

The selected model is the one with the highest PR-AUC on the chronological test set. In this notebook version, predictive performance is treated as a supporting step; the main study focus is the evaluation of LLM-generated explanations across diverse failure reasons.


In [ ]:
best_model_name = results_df.index[0]
best_model = fitted_models[best_model_name]
best_threshold = float(results_df.loc[best_model_name, "Threshold"])

print("Selected model:", best_model_name)
print("Selected threshold:", round(best_threshold, 4), "[train-only diagnostic threshold]")
display(results_df.loc[[best_model_name]].round(4))


## 9. Save compact prediction table for later XAI and explanation analysis


In [ ]:
best_scores = get_failure_scores(best_model, X_test)
best_pred = (best_scores >= best_threshold).astype(int)

prediction_table = test_df.copy()
prediction_table["predicted_PassOrFail"] = best_pred
prediction_table["failure_score"] = best_scores
prediction_table["error_type"] = np.select(
    [
        (prediction_table[TARGET] == 1) & (prediction_table["predicted_PassOrFail"] == 0),
        (prediction_table[TARGET] == 0) & (prediction_table["predicted_PassOrFail"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

prediction_table.to_csv("kamp_test_predictions_for_xai.csv", index=False)
print("Saved: kamp_test_predictions_for_xai.csv")
print(prediction_table["error_type"].value_counts())
print("\nFailure reasons available for explanation evaluation:")
display(prediction_table.loc[prediction_table[TARGET] == 1, "Reason"].value_counts(dropna=False).to_frame("count"))

prediction_table[["TimeStamp", TARGET, "Reason", "predicted_PassOrFail", "failure_score", "error_type"]].head()


In [ ]:
prediction_table

## Short interpretation guide

Use this notebook only as the predictive front-end:

- choose the best model by **PR-AUC**;
- inspect **Fail Recall** to see whether failures are actually being caught;
- inspect **Fail F1** to check whether the threshold gives a reasonable balance between missed failures and false alarms;
- pass `best_model`, `X_test`, `y_test`, and `prediction_table` into the next XAI/causality notebook.


In [ ]:
ax = results_df[["PR_AUC", "Fail_Recall", "Fail_F1"]].plot(
    kind="bar",
    figsize=(9, 5),
    rot=0
)

ax.set_title("Model Comparison on Top 3 Metrics")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(
    ["PR-AUC", "Fail Recall", "Fail F1"],
    title="Metric"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

<br> <br> <br>

## Feature Importance

In [ ]:
def get_top_n_xai_for_selected_failure(
    pipeline,
    X_train,
    X_test,
    y_test,
    selected_index,
    top_n=10,
    xai_method="shap",
    require_failure=True,
    random_state=None
):
    """
    Manually select one row from X_test using selected_index,
    apply either SHAP or LIME local feature importance,
    and return:

    1. xai_df:
        - feature_name
        - feature_importance

    2. selected_row_df:
        - the original selected row from X_test
        - true PassOrFail
        - predicted probability of Fail
        - selected XAI method
        - selected index

    Parameters
    ----------
    selected_index : index label from X_test
        The row index that should be explained.

    require_failure : bool, default=True
        If True, the selected row must have y_test == 1.
        If False, the function can explain any row.
    """

    xai_method = xai_method.lower()

    if xai_method not in ["shap", "lime"]:
        raise ValueError("xai_method must be either 'shap' or 'lime'.")

    # Ensure y_test is a pandas Series aligned with X_test
    if not isinstance(y_test, pd.Series):
        y_test = pd.Series(y_test, index=X_test.index)

    # 1. Validate manually selected row
    if selected_index not in X_test.index:
        raise ValueError(
            f"selected_index={selected_index} was not found in X_test.index."
        )

    if selected_index not in y_test.index:
        raise ValueError(
            f"selected_index={selected_index} was not found in y_test.index."
        )

    if require_failure and y_test.loc[selected_index] != 1:
        raise ValueError(
            f"selected_index={selected_index} is not a failure row. "
            f"y_test value is {y_test.loc[selected_index]}."
        )

    X_single = X_test.loc[[selected_index]]

    # 2. Extract preprocessing and model
    preprocessor = pipeline.named_steps["preprocess"]
    model = pipeline.named_steps["model"]

    # 3. Transform train and selected test row
    X_train_transformed = preprocessor.transform(X_train)
    X_single_transformed = preprocessor.transform(X_single)

    if hasattr(X_train_transformed, "toarray"):
        X_train_transformed = X_train_transformed.toarray()

    if hasattr(X_single_transformed, "toarray"):
        X_single_transformed = X_single_transformed.toarray()

    # 4. Get transformed feature names
    feature_names = preprocessor.get_feature_names_out()

    feature_names = [
        name.replace("num__", "").replace("cat__", "")
        for name in feature_names
    ]

    # 5. Predicted failure probability for selected row
    fail_probability = model.predict_proba(X_single_transformed)[0, 1]

    # 6. Store the selected original row
    selected_row_df = X_single.copy()
    selected_row_df["true_PassOrFail"] = int(y_test.loc[selected_index])
    selected_row_df["predicted_fail_probability"] = fail_probability
    selected_row_df["xai_method"] = xai_method
    selected_row_df["selected_index"] = selected_index

    # ============================================================
    # SHAP
    # ============================================================
    if xai_method == "shap":
        import shap

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_single_transformed)

        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        shap_values = np.array(shap_values)

        if len(shap_values.shape) == 3:
            shap_values = shap_values[:, :, 1]

        shap_values = shap_values.reshape(-1)

        xai_df = (
            pd.DataFrame({
                "feature_name": feature_names,
                "feature_importance": np.abs(shap_values)
            })
            .sort_values("feature_importance", ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )

    # ============================================================
    # LIME
    # ============================================================
    else:
        from lime.lime_tabular import LimeTabularExplainer

        explainer = LimeTabularExplainer(
            training_data=X_train_transformed,
            feature_names=feature_names,
            class_names=["Pass", "Fail"],
            mode="classification",
            discretize_continuous=True,
            random_state=random_state
        )

        explanation = explainer.explain_instance(
            data_row=X_single_transformed[0],
            predict_fn=model.predict_proba,
            num_features=top_n,
            labels=[1]
        )

        lime_map = explanation.as_map()[1]

        xai_df = (
            pd.DataFrame({
                "feature_name": [feature_names[i] for i, weight in lime_map],
                "feature_importance": [abs(weight) for i, weight in lime_map]
            })
            .sort_values("feature_importance", ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )

    return xai_df, selected_row_df

In [ ]:
X_test[y_test == 1]

In [44]:
# failed_x_test_rows = X_test.loc[test_df.loc[test_df["Reason"].notna()].index]
# failed_x_test_rows

test_df[test_df["Reason"].notna()]["Reason"]

2163       미성형
2177       미성형
2450    초기허용불량
2452    초기허용불량
2451    초기허용불량
2454    초기허용불량
2453    초기허용불량
4726       미성형
5183        가스
5185        가스
5188       미성형
5190       미성형
5192        가스
5205        가스
5207        가스
Name: Reason, dtype: str

## Note


- 가스 => Gas <br> 미성형 => Unshaped <br> 초기허용불량 => Initial Allowable Defect <br>

- Initial allowable or startup defects (`초기허용불량`) are treated as a trivial and expected failure mode, because they correspond to the initial production samples, where defects are anticipated during process stabilization; therefore, they are not considered a meaningful target for detailed root-cause explanation

- Rows with ids 2,3,4,5,6 will not be processed


In [82]:
test_df[test_df["Reason"].notna()]["Reason"]

2163       미성형
2177       미성형
2450    초기허용불량
2452    초기허용불량
2451    초기허용불량
2454    초기허용불량
2453    초기허용불량
4726       미성형
5183        가스
5185        가스
5188       미성형
5190       미성형
5192        가스
5205        가스
5207        가스
Name: Reason, dtype: str

In [124]:
# 14

In [ ]:
row_number = 14

row_reason = test_df[test_df["Reason"].notna()]["Reason"].iloc[row_number]
mapping = {"가스": "가스 => Gas", "미성형": "미성형 => Unshaped"}
row_reason_md = mapping.get(row_reason, row_reason)


failed_x_test_rows = X_test.loc[test_df.loc[test_df["Reason"].notna()].index]
index = int(failed_x_test_rows.iloc[row_number].to_frame().T.index[0])


top_n_shap_df, selected_failure_row_df = get_top_n_xai_for_selected_failure(
    pipeline=best_model,
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    selected_index=index,
    top_n=10,
    xai_method="shap",
    require_failure=True,
    # random_state=42
)


display(top_n_shap_df)
display(failed_x_test_rows.iloc[row_number].to_frame().T)


cols = [] # all rows
cols = ["Reason"]

# This is the equivalent row from df. They have different index since the data is diplaced a little bit 
# based on the fact that we took several rows from X_test for df["Reasons"]
display(df.loc[df["_id"] == test_df.loc[index, "_id"], "Reason"])


,feature_name,feature_importance
0,"PART_NAME_RG3 MOLD'G W/SHLD, LH",2.166576
1,Max_Back_Pressure,1.237193
2,Average_Screw_RPM,1.108703
3,Clamp_Close_Time,0.341588
4,Barrel_Temperature_5,0.338849
5,Plasticizing_Time,0.262676
6,Max_Injection_Pressure,0.220152
7,Barrel_Temperature_3,0.184678
8,Injection_Time,0.163124
9,Barrel_Temperature_6,0.097165


,PART_NAME,EQUIP_CD,EQUIP_NAME,Injection_Time,Filling_Time,Plasticizing_Time,Cycle_Time,Clamp_Close_Time,Cushion_Position,Switch_Over_Position,Plasticizing_Position,Clamp_Open_Position,Max_Injection_Speed,Max_Screw_RPM,Average_Screw_RPM,Max_Injection_Pressure,Max_Switch_Over_Pressure,Max_Back_Pressure,Average_Back_Pressure,Barrel_Temperature_1,Barrel_Temperature_2,Barrel_Temperature_3,Barrel_Temperature_4,Barrel_Temperature_5,Barrel_Temperature_6,Barrel_Temperature_7,Hopper_Temperature,Mold_Temperature_3,Mold_Temperature_4
5207,"RG3 MOLD'G W/SHLD, LH",S14,650톤-우진2호기,1.07,0.94,13.13,61.759998,6.79,654.23999,0.0,53.599998,4.63,127.400002,30.799999,29.0,143.0,118.699997,57.5,61.900002,285.799988,284.799988,284.899994,275.299988,265.0,235.199997,0.0,62.599998,22.5,24.4


5207    가스
Name: Reason, dtype: str

In [126]:
# Read the top_n_shap_df and append it as a markdown table to the explanation results file

# The kernel cwd may be either the notebooks/ folder or the project root,
# so pick whichever candidate directory actually exists (same pattern as DATA_PATH_CANDIDATES).
LLM_EXPLANATION_DIR = next(
    (p for p in [Path("../llm_explanation"), Path("llm_explanation")] if p.is_dir()),
    None,
)
if LLM_EXPLANATION_DIR is None:
    raise FileNotFoundError(
        "Could not find the llm_explanation directory. "
        f"Kernel cwd is {Path.cwd()} - update the candidate list."
    )

file_path = LLM_EXPLANATION_DIR / "explanation_results_sikdd.md"

table_markdown = top_n_shap_df.to_markdown(index=False)

# Create a section header and append the table
markdown_content = (
    f"\n\n## Top 10 SHAP Features for Index Row {int(index)}\n\n"
    f"{table_markdown}\n\n"
)

with open(file_path, "a", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Successfully appended top_n_shap_df table for Index Row {int(index)} to {file_path}")

Successfully appended top_n_shap_df table for Index Row 5207 to ..\llm_explanation\explanation_results_sikdd.md


<br> <br> <br>

## Generate an Expalnation Using LLM


In [127]:
feature_description_dict = {
    "PART_NAME": "The name or type of product part being manufactured.",
    "EQUIP_CD": "The machine or equipment code used to identify the production equipment.",
    "EQUIP_NAME": "The name of the machine or equipment used during production.",
    "Injection_Time": "The time taken to inject molten plastic into the mold.",
    "Filling_Time": "The time required to fill the mold cavity with material.",
    "Plasticizing_Time": "The time needed to melt and prepare plastic material before injection.",
    "Cycle_Time": "The total time required to complete one production cycle.",
    "Clamp_Close_Time": "The time taken for the mold clamp to close before injection starts.",
    "Cushion_Position": "The remaining screw position after injection, indicating leftover material in the barrel.",
    "Switch_Over_Position": "The screw position where the machine changes from injection control to pressure holding.",
    "Plasticizing_Position": "The screw position reached while preparing molten plastic for the next cycle.",
    "Clamp_Open_Position": "The mold opening position after the part has been formed.",
    "Max_Injection_Speed": "The highest speed reached while injecting material into the mold.",
    "Max_Screw_RPM": "The maximum rotation speed of the screw during plasticizing.",
    "Average_Screw_RPM": "The average screw rotation speed during material preparation.",
    "Max_Injection_Pressure": "The highest pressure applied while injecting material into the mold.",
    "Max_Switch_Over_Pressure": "The pressure at the moment the machine switches from injection to holding pressure.",
    "Max_Back_Pressure": "The highest resistance pressure applied during screw recovery and plasticizing.",
    "Average_Back_Pressure": "The average resistance pressure during material preparation.",
    "Barrel_Temperature_1": "The temperature in the first heating zone of the injection barrel.",
    "Barrel_Temperature_2": "The temperature in the second heating zone of the injection barrel.",
    "Barrel_Temperature_3": "The temperature in the third heating zone of the injection barrel.",
    "Barrel_Temperature_4": "The temperature in the fourth heating zone of the injection barrel.",
    "Barrel_Temperature_5": "The temperature in the fifth heating zone of the injection barrel.",
    "Barrel_Temperature_6": "The temperature in the sixth heating zone of the injection barrel.",
    "Barrel_Temperature_7": "The temperature in the seventh heating zone of the injection barrel.",
    "Hopper_Temperature": "The temperature around the hopper where raw plastic material enters the machine.",
    "Mold_Temperature_3": "The temperature measured in one monitored zone of the mold.",
    "Mold_Temperature_4": "The temperature measured in another monitored zone of the mold."
}

In [128]:
top_n_shap_df

,feature_name,feature_importance
0,"PART_NAME_RG3 MOLD'G W/SHLD, LH",2.166576
1,Max_Back_Pressure,1.237193
2,Average_Screw_RPM,1.108703
3,Clamp_Close_Time,0.341588
4,Barrel_Temperature_5,0.338849
5,Plasticizing_Time,0.262676
6,Max_Injection_Pressure,0.220152
7,Barrel_Temperature_3,0.184678
8,Injection_Time,0.163124
9,Barrel_Temperature_6,0.097165


In [129]:
from openai import OpenAI


def _extract_text_from_openrouter_response(response):
    """
    Extract the final assistant text from an OpenRouter/OpenAI-compatible response.

    Only reads `message.content` -- intentionally does NOT fall back to
    `message.reasoning` or `reasoning_details`, since those are the model's
    chain-of-thought, not its final answer. If `content` is empty, the caller
    should raise so the user can diagnose (usually max_tokens too low for a
    reasoning model).
    """

    try:
        message = response.choices[0].message
        content = getattr(message, "content", None)

        if isinstance(content, str) and content.strip():
            return content.strip()

        if isinstance(content, list):
            text_parts = []

            for part in content:
                if isinstance(part, dict):
                    if "text" in part and part["text"]:
                        text_parts.append(part["text"])
                    elif "content" in part and part["content"]:
                        text_parts.append(part["content"])
                else:
                    part_text = getattr(part, "text", None)
                    if part_text:
                        text_parts.append(part_text)

            final_text = "\n".join(text_parts).strip()
            if final_text:
                return final_text

    except Exception:
        pass

    try:
        raw = response.model_dump()
        message = raw["choices"][0]["message"]
        content = message.get("content")

        if isinstance(content, str) and content.strip():
            return content.strip()

        if isinstance(content, list):
            text_parts = []

            for part in content:
                if isinstance(part, dict):
                    if "text" in part and part["text"]:
                        text_parts.append(part["text"])
                    elif "content" in part and part["content"]:
                        text_parts.append(part["content"])

            final_text = "\n".join(text_parts).strip()
            if final_text:
                return final_text

    except Exception:
        pass

    return None


def generate_failure_explanation_with_llm(
    feature_description_dict,
    top_n_shap_df,
    llm_model="openai/gpt-5.4-mini",
    target_name="PassOrFail",
    max_rows=10,
    max_tokens=2000,
    reasoning_effort="low",
    return_prompt_only=False,
    debug=False,
):
    """
    Generate a short user-friendly explanation of a manufacturing failure
    using the top-N local SHAP feature importances of a single predicted
    failure together with plain-language feature descriptions.

    Parameters
    ----------
    top_n_shap_df : pd.DataFrame
        Must contain the columns "feature_name" and "feature_importance"
        (absolute SHAP values for one explained failure row).

    reasoning_effort : {"low", "medium", "high", None}
        For reasoning models (gpt-5-mini, o-series). "low" keeps internal
        reasoning short; `exclude=True` is always sent so the chain-of-thought
        is not returned. Pass `None` for non-reasoning models -- nothing
        reasoning-related is sent.
    max_tokens : int
        Total completion budget. For reasoning models, this must cover the
        model's internal reasoning AND the final answer. Keep >= 2000 for
        gpt-5-mini even with `reasoning_effort="low"`.

    Returns
    -------
    explanation : str or None
        LLM-generated explanation. If return_prompt_only=True, returns None.

    prompt_payload : dict
        Full prompt and metadata before the LLM call.
    """

    load_dotenv()

    api_key = os.getenv("OPENROUTER_API_KEY")

    if not api_key and not return_prompt_only:
        raise ValueError("OPENROUTER_API_KEY was not found in the .env file.")

    if top_n_shap_df is None or top_n_shap_df.empty:
        raise ValueError("top_n_shap_df is empty. No SHAP rows were provided.")

    required_cols = {"feature_name", "feature_importance"}
    missing_cols = required_cols - set(top_n_shap_df.columns)

    if missing_cols:
        raise ValueError(f"top_n_shap_df is missing columns: {missing_cols}")

    shap_rows = (
        top_n_shap_df
        .sort_values("feature_importance", ascending=False)
        .head(max_rows)
        .copy()
    )

    shap_rows["feature_description"] = shap_rows["feature_name"].map(
        feature_description_dict
    ).fillna(shap_rows["feature_name"])

    shap_records = shap_rows[
        [
            "feature_name",
            "feature_description",
            "feature_importance",
        ]
    ].to_dict(orient="records")

    system_prompt = """
You explain manufacturing failures to production and quality-control users.

The input comes from a local feature-importance analysis of a machine-learning
model that predicted a single production cycle as a failure. Each listed
process variable has an importance score showing how strongly it contributed
to the failure prediction for this specific cycle. The scores show strength of
contribution only, not direction, and they are not definitive proof of
causality. Explain them as possible contributing factors or associated process
conditions.

Rules:
- Keep the explanation short and clear.
- Maximum length: 5 sentences.
- Use simple manufacturing language.
- Prefer the feature descriptions instead of raw feature names.
- The variables are listed from most influential to least influential; give more weight to the most influential ones.
- Describe each variable as a possible contributor to failure risk, not a proven cause.
- Do not claim that a variable was too high or too low; the importance scores do not carry direction.
- Do not overclaim.
- Do not mention SHAP, LIME, machine learning, models, importance scores, or coefficients to the end user.
- Return only the final explanation as plain text. Do not include reasoning, preambles, or labels such as "Explanation:".
"""

    user_prompt = f"""
Target variable:
{target_name}

Feature descriptions:
{json.dumps(feature_description_dict, indent=2)}

Top process variables that contributed to this failure prediction, most influential first:
{json.dumps(shap_records, indent=2)}

Write one short explanation for the end user about why the failure may have happened.
Return only the explanation text.
"""

    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()},
    ]

    extra_body = None
    if reasoning_effort is not None:
        extra_body = {
            "reasoning": {
                "effort": reasoning_effort,
                "exclude": True,
            }
        }

    prompt_payload = {
        "model": llm_model,
        "messages": messages,
        "system_prompt": system_prompt.strip(),
        "user_prompt": user_prompt.strip(),
        "shap_records": shap_records,
        "extra_body": extra_body,
        "max_tokens": max_tokens,
    }

    if return_prompt_only:
        return None, prompt_payload

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )

    create_kwargs = {
        "model": llm_model,
        "messages": messages,
        "max_tokens": max_tokens,
    }
    if extra_body is not None:
        create_kwargs["extra_body"] = extra_body

    response = client.chat.completions.create(**create_kwargs)

    if debug:
        print(response.model_dump_json(indent=2))

    explanation = _extract_text_from_openrouter_response(response)

    if explanation is None:
        prompt_payload["raw_response"] = response.model_dump()

        raise ValueError(
            "The LLM response did not contain extractable final text. "
            "For reasoning models (gpt-5-mini, o-series), this usually means "
            "max_tokens was too low: internal reasoning consumed the entire "
            "budget and no `content` was produced. Try increasing max_tokens "
            "(e.g. 4000) or lowering reasoning_effort. "
            "Run with debug=True and inspect prompt_payload['raw_response'] to confirm."
        )

    return explanation, prompt_payload

In [130]:
# The kernel cwd may be either the notebooks/ folder or the project root,
# so pick whichever candidate directory actually exists (same pattern as DATA_PATH_CANDIDATES).

file_path = "../llm_explanation/explanation_results_sikdd.md"

llm_models = [
    "openai/gpt-5.4-mini",
    "google/gemini-3.1-flash-lite",
    "mistralai/devstral-2512",
]

explanations = {}

for llm_model in llm_models:
    
    explanation, prompt_payload = generate_failure_explanation_with_llm(
        feature_description_dict=feature_description_dict,
        top_n_shap_df=top_n_shap_df,
        llm_model=llm_model,
        target_name="PassOrFail",
        max_rows=10,
        max_tokens=2000,
        reasoning_effort="medium",
        debug=False,
    )

    explanations[llm_model] = explanation

    # --- FILE APPENDING LOGIC ---
    # Format the results into a single Markdown string
    markdown_content = (
        f"{'-'*100}\n"
        f"**Model:** `{llm_model}`\n\n"
        f"**Index Row:** {int(index)}\n\n"
        f"**Reason:** {row_reason_md}\n\n"        
        f"{explanation}\n\n"
    )

    # Open the file in append mode ('a') and write the content
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(markdown_content)

    print(f"Successfully appended results for Index Row {int(index)} to {file_path}")

    print("-"*200)
    display(Markdown(f"**Model:** `{llm_model}`"))
    display(Markdown(f"**Index Row:** {int(index)}"))
    display(Markdown(explanation))

# Close the section for this row so consecutive rows stay visually separated
with open(file_path, "a", encoding="utf-8") as f:
    f.write(f"{'-'*100}\n{'-'*100}\n{'-'*100}\n")

Successfully appended results for Index Row 5207 to ../llm_explanation/explanation_results_sikdd.md
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


**Model:** `openai/gpt-5.4-mini`

**Index Row:** 5207

This failure may be linked first to the specific part being run, which can have different process sensitivity than other parts. The strongest process-related factors were the screw recovery and plasticizing conditions, especially the highest back pressure and the average screw speed, which may have affected how the material was prepared. Clamp close time, barrel temperatures in several zones, plasticizing time, injection pressure, and injection time also stood out as possible contributors, suggesting the cycle may have had unstable molding conditions.

Successfully appended results for Index Row 5207 to ../llm_explanation/explanation_results_sikdd.md
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


...

The production of this specific part type was the most notable factor associated with this cycle's performance. The variation in back pressure and the average screw rotation speed during material preparation also appear to have contributed to the risk of failure. Additionally, fluctuations in clamp closing time and local temperature settings within the injection barrel may have influenced this outcome. Together, these factors suggest that settings related to material preparation and the initial injection stage were significant contributors to this production issue. Please review these process parameters to ensure they remain consistent with established operating standards.

Successfully appended results for Index Row 5207 to ../llm_explanation/explanation_results_sikdd.md
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


**Model:** `mistralai/devstral-2512`

**Index Row:** 5207

This failure may be linked to the specific part being produced (PART_NAME_RG3 MOLD'G W/SHLD, LH), as it was the most strongly associated factor. The highest resistance pressure during screw recovery and plasticizing, along with the average screw rotation speed during material preparation, also played a notable role. Additionally, variations in mold clamp closing time, barrel temperatures in zones 5 and 3, and plasticizing time may have contributed to the issue. Injection pressure and injection time were less influential but could still be related. These conditions together suggest a combination of part-specific and process factors that may have increased failure risk.